In this notebook we detect trees and extract their heights.

Inputs: NDVI, DTM and DEM rasters

Processing steps (for every DSM file):
- calculate height of objects: DEM-DTM
- make a tree mask considering NDVI > 0.2 and height >2m
- mask the height raster: get a new height raster with only trees height
- reduce raster noise: apply max, median and mean filters
- simplify raster: make "blocks" every 1.5m
- extract the polygons for each different height of the raster
- saves a gdf with the results: 50 gdf cover all the city

Cleaning steps (for every gdf saved):
- remove polygons with height 0
- remove polygons with area <= 1.5m
- assigns CRS 4326 (neccesary for using pybdshadow library)
- explode multypoligons if there are
- simplify and smooth geometries

note: during all the steps, the temporary and previous files are removed to avoid memory issues (both disk and ram). Also every 20 iterations there is a sleep(30) to avoid memory overload

In [ ]:
!pip install geopandas shapely rasterio


In [1]:
# Built-ins
import os
import io
import json
import math
import time
import glob

# Third-party
import numpy as np
import pandas as pd
from tqdm import tqdm
import boto3

# Geospatial
import geopandas as gpd
from shapely.geometry import box, shape, Polygon, MultiPolygon

# Raster I/O
import rasterio
from rasterio.crs import CRS
from rasterio.mask import mask
from rasterio.features import shapes
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds

# Filters / image processing
from scipy.ndimage import maximum_filter, median_filter, uniform_filter



In [2]:
# INPUT FOLDER WITH TREE HEIGHT RASTERS
rasters_folder_path = "../../../../datos-notebooks/2.global-data-bologna/2.2.objects-height/gch_bologna"
clipped_rasters_folder_path = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/clipped-rasters/"
trees_vectors_path = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors/"
clean_trees_vectors_path = "../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors-clean/"
bound_path = "../../../../datos-notebooks/bologne_boundary.geojson"

In [10]:
# Clip to city boundary
os.makedirs(clipped_rasters_folder_path, exist_ok=True)

a = os.path.join(rasters_folder_path,'meta_canopy_tile_0.tif')
with rasterio.open(a) as src:
    crs = src.crs
    
bounds = gpd.read_file(bound_path)
bounds = bounds.to_crs(crs) # Ensure CRS is projected and in same crs 
bounds_buffer = bounds.buffer(100) # Buffer 100 meters


for file in os.listdir(rasters_folder_path):

    raster_path = os.path.join(rasters_folder_path, file)

    with rasterio.open(raster_path) as src:

        raster_bounds = box(*src.bounds) # Get raster bounding box as a polygon
        raster_gdf = gpd.GeoDataFrame(geometry=[raster_bounds], crs=src.crs)

        # CHECK IF RASTER INTERSECTS THE BUFFER AREA
        if not raster_gdf.intersects(bounds_buffer.geometry.union_all()).iloc[0]:

            print(f"Skipping {file} (no intersection).")
            continue

        print(f"Clipping {file} ...")

        # CLIP THE RASTER
        # Convert geometry to GeoJSON-like mapping for rasterio.mask
        shapes = [bounds_buffer.geometry.union_all().__geo_interface__]

        try:
            out_img, out_transform = mask(src, shapes, crop=True)
        except Exception as e:
            print(f"Error clipping {file}: {e}")
            continue

        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_img.shape[1],
            "width": out_img.shape[2],
            "transform": out_transform
        })

        # SAVE CLIPPED RASTER
        out_path = os.path.join(clipped_rasters_folder_path, f"clipped_{file}")

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(out_img)

        print(f"Saved → {out_path}")


Skipping meta_canopy_tile_0.tif (no intersection).
Skipping meta_canopy_tile_1.tif (no intersection).
Clipping meta_canopy_tile_10.tif ...
Saved → ../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/clipped-rasters/clipped_meta_canopy_tile_10.tif
Skipping meta_canopy_tile_11.tif (no intersection).
Skipping meta_canopy_tile_12.tif (no intersection).
Skipping meta_canopy_tile_13.tif (no intersection).
Skipping meta_canopy_tile_14.tif (no intersection).
Skipping meta_canopy_tile_15.tif (no intersection).
Skipping meta_canopy_tile_2.tif (no intersection).
Skipping meta_canopy_tile_3.tif (no intersection).
Clipping meta_canopy_tile_4.tif ...
Saved → ../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/clipped-rasters/clipped_meta_canopy_tile_4.tif
Clipping meta_canopy_tile_5.tif ...
Saved → ../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/clipped-rasters/clipped_meta_canopy_tile_5.tif
Clipping met

In [3]:
# CONFIGURATION
height_list = sorted(glob.glob(os.path.join(clipped_rasters_folder_path, "*.tif")))
os.makedirs(trees_vectors_path, exist_ok=True)
processed_file_list = "processed_files.json" # to follow the processing progress

In [ ]:
# Load already processed files
if os.path.exists(processed_file_list):
    with open(processed_file_list, "r") as f:
        processed_files = set(json.load(f))
else:
    processed_files = set()

# Main loop: extract vectors of trees
for raster_path in tqdm(height_list):

    filename = os.path.basename(raster_path)

    # Skip files already processed
    if filename in processed_files:
        continue

    try:
        # Read raster
        with rasterio.open(raster_path) as src:
            height = src.read(1)
            transform = src.transform
            crs = src.crs

        height = np.where(height > 2, height, 0)  # Remove low heights

        # Apply smoothing filters
        h1 = maximum_filter(height, size=3)
        h2 = median_filter(h1, size=7)
        h3 = uniform_filter(h2, size=2)

        # Quantize heights
        height = np.round(h3 / 1.5) * 1.5

        # delete intermediates to free memory
        del h1, h2, h3

        # Extract polygons (vectorization)
        results = []

        for geom, val in shapes(
            height.astype(np.int16),     # raster as int16
            transform=transform
        ):
            if val > 0:
                results.append({
                    "height": float(val),
                    "geometry": shape(geom),
                    "filename": filename
                })

        # Build GeoDataFrame
        gdf = gpd.GeoDataFrame(results, geometry="geometry", crs=crs)

        # Save one output per raster
        out_file = os.path.join(trees_vectors_path, f"{filename}.json")  
        # save file directly without grouping
        gdf.to_file(out_file, driver="GeoJSON")

        # Register processed file
        processed_files.add(filename)
        with open(processed_file_list, "w") as f:
            json.dump(sorted(processed_files), f)

        del height, gdf, results  # Free memory explicitly

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        continue



100%|██████████| 6/6 [30:49<00:00, 308.19s/it]


In [ ]:
# Clean up
os.remove('processed_files.json')

In [14]:
# functions for smoothing results

def chaikin_smoothing(coords, refinements=2):
    for _ in range(refinements):
        new_coords = []
        for i in range(len(coords) - 1):
            p0 = np.array(coords[i])
            p1 = np.array(coords[i + 1])
            Q = 0.75 * p0 + 0.25 * p1
            R = 0.25 * p0 + 0.75 * p1
            new_coords.extend([Q, R])
        new_coords.append(new_coords[0])  # cerrar el anillo
        coords = new_coords
    return coords

def smooth_polygon(poly, refinements=2):
    if isinstance(poly, Polygon):
        smoothed = Polygon(chaikin_smoothing(list(poly.exterior.coords), refinements))
        return smoothed
    elif isinstance(poly, MultiPolygon):
        return MultiPolygon([smooth_polygon(p, refinements) for p in poly.geoms])
    else:
        return poly


In [ ]:
import os
import geopandas as gpd
import json

# Spliting big files

# Folder where the previous step saved the per-raster GeoJSON files
folder = trees_vectors_path  
max_size = 300 * 1024  # 300 KB in bytes

# Loop through all GeoJSON files
for file in os.listdir(folder):

    if not file.endswith(".json"):
        continue

    full_path = os.path.join(folder, file)
    size_bytes = os.path.getsize(full_path)

    # Skip small files
    if size_bytes <= max_size:
        continue
    
    print(f"Splitting large file: {file} ({size_bytes/1024:.1f} KB)")

    # Load file
    gdf = gpd.read_file(full_path)

    # Start splitting
    part_id = 1
    temp_gdf = gpd.GeoDataFrame(columns=gdf.columns, crs=gdf.crs)

    for idx, row in gdf.iterrows():

        # Append row
        temp_gdf = temp_gdf._append(row, ignore_index=True)

        # Save test file to check size
        test_path = os.path.join(folder, f"{file[:-5]}_temp.json")
        temp_gdf.to_file(test_path, driver="GeoJSON")

        # If temp file exceeds 300 KB, save previous part and reset
        if os.path.getsize(test_path) > max_size:

            # Remove the last added row (it caused overflow)
            temp_gdf = temp_gdf.iloc[:-1]

            # Save the valid part
            out_path = os.path.join(folder, f"{file[:-5]}_part{part_id}.json")
            temp_gdf.to_file(out_path, driver="GeoJSON")  # save part
            print(f" → Saved {out_path}")

            part_id += 1

            # Start new part with the row that overflowed
            temp_gdf = gpd.GeoDataFrame([row], columns=gdf.columns, crs=gdf.crs)

        # Remove test file
        os.remove(test_path)

    # Save the last part
    final_path = os.path.join(folder, f"{file[:-5]}_part{part_id}.json")
    temp_gdf.to_file(final_path, driver="GeoJSON")
    print(f" → Saved {final_path}")



In [ ]:
# Cleaning process for all split GeoJSONs
os.makedirs(clean_trees_vectors_path, exist_ok=True)

# Loop through every JSON file in the folder
for file in tqdm(os.listdir(trees_vectors_path)):

    if not file.endswith(".json"):
        continue

    part_file = os.path.join(trees_vectors_path, file)

    # Load partial GeoJSON
    gdf = gpd.read_file(part_file)

    # Force CRS and clean geometry
    gdf = gdf.set_crs(32632, allow_override=True)    # overwrite CRS
    gdf = gdf[gdf['height'] != 0]                    # remove height=0
    gdf = gdf.explode(index_parts=False)             # explode geometries
    
    time.sleep(5)                                    # cooldown
    
    # smooth and simplify
    gdf['geometry'] = gdf['geometry'].apply(
        lambda geom: smooth_polygon(geom, refinements=5)
    )
    gdf['geometry'] = gdf['geometry'].simplify(
        tolerance=0.15, 
        preserve_topology=True
    )

    # filter invalid or tiny geometries
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[gdf.geometry.area >= 1.5]
    gdf['geometry'] = gdf['geometry'].buffer(0)      # fix topology

    # final CRS
    gdf = gdf.to_crs("EPSG:4326")

    # Output filename
    clean_name = file.replace(".json", "_clean.geojson")
    clean_path = os.path.join(clean_trees_vectors_path, clean_name)

    # Save result
    gdf.to_file(clean_path, driver="GeoJSON")

    # Optional: remove original partial file to free space
    # os.remove(part_file)


In [ ]:
# Cleaning process - 5h aprox -- borrar despues. el de arriba es el final con archivos mas chicos 
# Process each partial file individually and save the results in clean_results
os.makedirs(clean_trees_vectors_path, exist_ok=True)

#for j in tqdm(range(num_parts)):
for j in tqdm(range(21)): 
    part_file = os.path.join(trees_vectors_path, f"part{j+1}.json")
    if not os.path.exists(part_file):
        continue

    gdf = gpd.read_file(part_file)
    gdf = gdf.set_crs(32632, allow_override=True)
    gdf = gdf[gdf['height'] != 0]
    gdf = gdf.explode(index_parts=False)
    time.sleep(5)
    gdf['geometry'] = gdf['geometry'].apply(lambda geom: smooth_polygon(geom, refinements=5))
    gdf['geometry'] = gdf['geometry'].simplify(tolerance=0.15, preserve_topology=True)
    gdf = gdf[gdf.geometry.notnull()]
    gdf = gdf[gdf.geometry.area >= 1.5]
    gdf['geometry'] = gdf['geometry'].buffer(0)
    gdf = gdf.to_crs("EPSG:4326")
    clean_path = os.path.join(clean_trees_vectors_path, f"part{j+1}_clean.geojson")
    gdf.to_file(clean_path, driver="GeoJSON")

    #os.remove(part_file) #removes the input file to make some space in disk

100%|██████████| 21/21 [4:27:50<00:00, 765.27s/it]   


In [ ]:
import os
import geopandas as gpd

# demoró 110 min y no hizo nada

## solo por esta vez, dividir los resultados. despues el codigo ya va a quedar dividido -- borrar despues
folder = clean_trees_vectors_path
max_size = 300000 * 1024  # 300 KB

for file in os.listdir(folder):

    if not file.endswith(".geojson"):
        continue

    full_path = os.path.join(folder, file)
    size_bytes = os.path.getsize(full_path)

    if size_bytes <= max_size:
        continue

    print(f"Splitting large file: {file} ({size_bytes/1024:.1f} KB)")

    gdf = gpd.read_file(full_path)

    part_id = 1
    temp_gdf = gpd.GeoDataFrame(columns=gdf.columns, crs=gdf.crs)

    for idx, row in gdf.iterrows():

        # Convert row to GeoDataFrame before concatenation
        row_gdf = gpd.GeoDataFrame([row], columns=gdf.columns, crs=gdf.crs)

        # Add row using concat (append is deprecated)
        temp_gdf = pd.concat([temp_gdf, row_gdf], ignore_index=True)

        # Ensure CRS stays defined
        temp_gdf = temp_gdf.set_crs(gdf.crs, allow_override=True)

        # Temporary file to test size
        test_path = os.path.join(folder, f"{file[:-8]}_temp.geojson")
        temp_gdf.to_file(test_path, driver="GeoJSON")

        if os.path.getsize(test_path) > max_size:

            # Remove last row (overflow)
            temp_gdf = temp_gdf.iloc[:-1]
            temp_gdf = temp_gdf.set_crs(gdf.crs, allow_override=True)

            # Save previous valid part
            out_path = os.path.join(folder, f"{file[:-8]}_part{part_id}.geojson")
            temp_gdf.to_file(out_path, driver="GeoJSON")
            print(f" → Saved {out_path}")

            part_id += 1

            # Start new part with the overflowing row
            temp_gdf = row_gdf.copy()
            temp_gdf = temp_gdf.set_crs(gdf.crs, allow_override=True)

        os.remove(test_path)

    # Save last part
    final_path = os.path.join(folder, f"{file[:-8]}_part{part_id}.geojson")
    temp_gdf.to_file(final_path, driver="GeoJSON")
    print(f" → Saved {final_path}")

    os.remove(full_path)
    print(f"Deleted original: {file}\n")


Splitting large file: part1_clean.geojson (1254785.2 KB)


C:\Users\guada\AppData\Local\Temp\ipykernel_23772\1755536412.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  temp_gdf = pd.concat([temp_gdf, row_gdf], ignore_index=True)


KeyboardInterrupt: 